# ROC Curve & AUC Explorer

Chapter 13: Probability and Random Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

Sweep a classifier threshold to trace TPR vs FPR and read AUC.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-13/roc-auc/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(0)

# The demo's defaults: scores ~ N(1.2, 1) for positives, N(-0.3, 1) for negatives, threshold 0.5
mu_p, mu_n, sigma, t = 1.2, -0.3, 1.0, 0.5
N = torch.distributions.Normal(0.0, 1.0)


def rates(thr):
    tpr = 1 - N.cdf(torch.tensor((thr - mu_p) / sigma))
    fpr = 1 - N.cdf(torch.tensor((thr - mu_n) / sigma))
    return tpr.item(), fpr.item()


tpr, fpr = rates(t)
print(f"threshold {t}: TPR = {tpr:.4f}, FPR = {fpr:.4f}")

# ── Sweep the threshold to trace the ROC curve, area by the trapezoid rule ──
ts = torch.linspace(mu_n - 6 * sigma, mu_p + 6 * sigma, 2001)
tprs = 1 - N.cdf((ts - mu_p) / sigma)
fprs = 1 - N.cdf((ts - mu_n) / sigma)
auc_trap = torch.trapezoid(tprs.flip(0), fprs.flip(0)).item()
# For two equal-variance Gaussians the AUC has a closed form
auc_exact = N.cdf(torch.tensor((mu_p - mu_n) / (sigma * math.sqrt(2)))).item()
print(f"AUC (trapezoid) = {auc_trap:.5f},  AUC (closed form Phi(d'/sqrt 2)) = {auc_exact:.5f}")

# ── AUC = P(score of a random positive > score of a random negative) ────────
pos = mu_p + sigma * torch.randn(200000)
neg = mu_n + sigma * torch.randn(200000)
print("Monte Carlo P(pos > neg) =", (pos > neg).float().mean().item())

# Same AUC from ranks (Mann-Whitney U), the way sklearn's roc_auc_score works
scores = torch.cat([pos[:5000], neg[:5000]])
ranks = scores.argsort().argsort().double() + 1
u = ranks[:5000].sum() - 5000 * 5001 / 2
print("rank-based AUC on 5000 + 5000 samples =", (u / (5000 * 5000)).item())

threshold 0.5: TPR = 0.7580, FPR = 0.2119
AUC (trapezoid) = 0.85558,  AUC (closed form Phi(d'/sqrt 2)) = 0.85558
Monte Carlo P(pos > neg) = 0.854794979095459
rank-based AUC on 5000 + 5000 samples = 0.8544078
